In [1]:
from pathlib import Path
import pandas as pd

In [2]:
saved_dir = Path(r"C:\Users\nb0801\Documents\GitHub\IDS-CAN-Bus-In-Vehicle-Networks-Based-on-the-Statistical-Characteristics-of-Attacks\saved_data\ROAD")

train_dfs = pd.read_pickle(saved_dir / "train_dfs.pkl")
test_dfs = pd.read_pickle(saved_dir / "test_dfs.pkl")

print("Reloaded train_dfs and test_dfs from saved_data/")
print("Train dfs:", len(train_dfs))
print("Test dfs:", len(test_dfs))
print("Train sizes:", [len(df) for df in train_dfs])
print("Test sizes:", [len(df) for df in test_dfs])

Reloaded train_dfs and test_dfs from saved_data/
Train dfs: 38
Test dfs: 38
Train sizes: [41580, 39911, 42657, 40945, 25197, 24186, 8058, 8024, 49098, 47143, 62576, 60064, 121688, 116802, 13400, 12862, 47249, 45352, 48501, 46554, 39720, 38128, 73526, 70575, 46872, 44992, 2393018, 850900, 617216, 1139367, 632354, 746282, 91319, 3535538, 1257855, 98438, 899801, 7057533]
Test sizes: [10394, 9977, 10664, 10236, 6299, 6046, 2014, 2006, 12274, 11785, 15644, 15016, 30421, 29200, 3349, 3215, 11812, 11338, 12125, 11638, 9930, 9531, 18381, 17643, 11718, 11247, 598254, 212724, 154303, 284841, 158088, 186570, 22829, 883884, 314463, 24609, 224950, 1764383]


In [3]:
def add_context_features(df):
    """
    Adds neighbouring ID and timing features to a single capture.
    """

    df = df.sort_values("Timestamp").reset_index(drop=True)

    # Neighbour IDs
    df["Prev_ID"] = df["ID"].shift(1)
    df["Next_ID"] = df["ID"].shift(-1)
    df["Former_ID"] = df["ID"].shift(2)
    df["Latter_ID"] = df["ID"].shift(-2)

    # Adjacent packet intervals
    df["Prev_interval"] = df["Timestamp"] - df["Timestamp"].shift(1)
    df["Next_interval"] = df["Timestamp"].shift(-1) - df["Timestamp"]

    # Previous/next packet with the same ID
    df["_Prev_same_ts"] = df.groupby("ID")["Timestamp"].shift(1)
    df["_Next_same_ts"] = df.groupby("ID")["Timestamp"].shift(-1)

    df["Prev_same_ID_interval"] = (
        df["Timestamp"] - df["_Prev_same_ts"]
    )
    df["Next_same_ID_interval"] = (
        df["_Next_same_ts"] - df["Timestamp"]
    )

    df.drop(columns=["_Prev_same_ts", "_Next_same_ts"], inplace=True)

    return df


# ----------------------------------------------------
# Compute features for every capture individually
# ----------------------------------------------------

train_dfs = [add_context_features(df) for df in train_dfs]
test_dfs  = [add_context_features(df) for df in test_dfs]

# ----------------------------------------------------
# Merge afterwards
# ----------------------------------------------------

train_dfs = pd.concat(train_dfs, ignore_index=True)
test_dfs = pd.concat(test_dfs, ignore_index=True)

print("Merged train_dfs shape:", train_dfs.shape)
print("Merged test_dfs shape:", test_dfs.shape)

# Show new columns
print(train_dfs[[
    "ID",
    "Prev_ID",
    "Next_ID",
    "Former_ID",
    "Latter_ID",
    "Prev_interval",
    "Next_interval",
    "Prev_same_ID_interval",
    "Next_same_ID_interval"
]].head())

Merged train_dfs shape: (20535281, 13)
Merged test_dfs shape: (5133801, 13)
    ID Prev_ID Next_ID Former_ID Latter_ID  Prev_interval  Next_interval  \
0  5e1     NaN     28b       NaN       0d0            NaN   1.072884e-06   
1  28b     5e1     0d0       NaN       033   1.072884e-06   9.536743e-07   
2  0d0     28b     033       5e1       125   9.536743e-07   1.001954e-03   
3  033     0d0     125       28b       0a7   1.001954e-03   1.072884e-06   
4  125     033     0a7       0d0       2d2   1.072884e-06   9.536743e-07   

   Prev_same_ID_interval  Next_same_ID_interval  
0                    NaN               0.011871  
1                    NaN               0.020143  
2                    NaN               0.012873  
3                    NaN               0.012892  
4                    NaN               0.009813  


In [4]:
#Calculate Data Length Code (DLC) - number of bytes in the Data field
train_dfs["DLC"] = train_dfs["Data"].str.replace(" ", "").str.len() // 2
test_dfs["DLC"] = test_dfs["Data"].str.replace(" ", "").str.len() // 2

print("Train DLC added:")
print(train_dfs[["Data", "DLC"]].head())
print("\nTest DLC added:")
print(test_dfs[["Data", "DLC"]].head())

Train DLC added:
               Data  DLC
0  8935000B09FFEC80    8
1  0000000000000000    8
2  0A770460F1030700    8
3  000650000C4317D0    8
4  900040DF403F6F60    8

Test DLC added:
               Data  DLC
0  595945450000FFFF    8
1  6000000000000000    8
2  000803C3EA11F4CE    8
3  0010FA24C12E10A0    8
4  00000000001D0800    8


In [5]:
# sort by time and add neighbor ID columns and interval features
test_dfs = test_dfs.sort_values("Timestamp").reset_index(drop=True)

test_dfs["Prev_ID"] = test_dfs["ID"].shift(1)
test_dfs["Next_ID"] = test_dfs["ID"].shift(-1)
test_dfs["Former_ID"] = test_dfs["ID"].shift(2)   # before previous
test_dfs["Latter_ID"] = test_dfs["ID"].shift(-2)  # after next

test_dfs["Prev_interval"] = test_dfs["Timestamp"] - test_dfs["Timestamp"].shift(1)
test_dfs["Next_interval"] = test_dfs["Timestamp"].shift(-1) - test_dfs["Timestamp"]

# previous/next same-ID intervals
test_dfs["_Prev_same_ts"] = test_dfs.groupby("ID")["Timestamp"].shift(1)
test_dfs["_Next_same_ts"] = test_dfs.groupby("ID")["Timestamp"].shift(-1)

test_dfs["Prev_same_ID_interval"] = test_dfs["Timestamp"] - test_dfs["_Prev_same_ts"]
test_dfs["Next_same_ID_interval"] = test_dfs["_Next_same_ts"] - test_dfs["Timestamp"]

test_dfs.drop(columns=["_Prev_same_ts", "_Next_same_ts"], inplace=True)

# show the new columns
print(test_dfs[[
    "ID","Prev_ID","Next_ID","Former_ID","Latter_ID",
    "DLC","Prev_interval","Next_interval",
    "Prev_same_ID_interval","Next_same_ID_interval"
]].head())

    ID Prev_ID Next_ID Former_ID Latter_ID  DLC  Prev_interval  Next_interval  \
0  162     NaN     0A7       NaN       FFF    8            NaN   2.026558e-06   
1  0A7     162     FFF       NaN       498    8   2.026558e-06   9.536743e-07   
2  FFF     0A7     498       162       464    8   9.536743e-07   1.072884e-06   
3  498     FFF     464       0A7       577    8   1.072884e-06   1.012921e-03   
4  464     498     577       FFF       69D    8   1.012921e-03   1.072884e-06   

   Prev_same_ID_interval  Next_same_ID_interval  
0                    NaN               0.018992  
1                    NaN               0.006927  
2                    NaN               0.011905  
3                    NaN               0.019993  
4                    NaN               0.098999  


In [6]:
import pandas as pd

# Only keep the columns you need
X_train = train_dfs.loc[:, [
    "ID","Prev_ID","Next_ID","Former_ID","Latter_ID",
    "DLC","Prev_interval","Next_interval",
    "Prev_same_ID_interval","Next_same_ID_interval"
]].copy()


def parse_hex_id(value):
    if pd.isna(value):
        return pd.NA
    if isinstance(value, str):
        value = value.strip()
        if not value:
            return pd.NA
        if value.lower().startswith("0x"):
            value = value[2:]
        return int(value, 16)
    return int(value)

# Convert hexadecimal CAN IDs directly to nullable integers
for col in ["ID", "Prev_ID", "Next_ID", "Former_ID", "Latter_ID"]:
    X_train[col] = X_train[col].map(parse_hex_id).astype("UInt32")

# DLC only needs values 0-8
X_train["DLC"] = X_train["DLC"].astype("UInt8")


# Labels
y_train = train_dfs["Attack"][:].astype("category")

print(X_train.head())

     ID  Prev_ID  Next_ID  Former_ID  Latter_ID  DLC  Prev_interval  \
0  1505     <NA>      651       <NA>        208    8            NaN   
1   651     1505      208       <NA>         51    8   1.072884e-06   
2   208      651       51       1505        293    8   9.536743e-07   
3    51      208      293        651        167    8   1.001954e-03   
4   293       51      167        208        722    8   1.072884e-06   

   Next_interval  Prev_same_ID_interval  Next_same_ID_interval  
0   1.072884e-06                    NaN               0.011871  
1   9.536743e-07                    NaN               0.020143  
2   1.001954e-03                    NaN               0.012873  
3   1.072884e-06                    NaN               0.012892  
4   9.536743e-07                    NaN               0.009813  


In [7]:
import pandas as pd

# Only keep the columns you need
X_test = test_dfs.loc[:, [
    "ID","Prev_ID","Next_ID","Former_ID","Latter_ID",
    "DLC","Prev_interval","Next_interval",
    "Prev_same_ID_interval","Next_same_ID_interval"
]].copy()


def parse_hex_id(value):
    if pd.isna(value):
        return pd.NA
    if isinstance(value, str):
        value = value.strip()
        if not value:
            return pd.NA
        if value.lower().startswith("0x"):
            value = value[2:]
        return int(value, 16)
    return int(value)

# Convert hexadecimal CAN IDs directly to nullable integers
for col in ["ID", "Prev_ID", "Next_ID", "Former_ID", "Latter_ID"]:
    X_test[col] = X_test[col].map(parse_hex_id).astype("UInt32")

# DLC only needs values 0-8
X_test["DLC"] = X_test["DLC"].astype("UInt8")


# Labels
y_test = test_dfs["Attack"][:].astype("category")

print(X_test.head())

     ID  Prev_ID  Next_ID  Former_ID  Latter_ID  DLC  Prev_interval  \
0   354     <NA>      167       <NA>       4095    8            NaN   
1   167      354     4095       <NA>       1176    8   2.026558e-06   
2  4095      167     1176        354       1124    8   9.536743e-07   
3  1176     4095     1124        167       1399    8   1.072884e-06   
4  1124     1176     1399       4095       1693    8   1.012921e-03   

   Next_interval  Prev_same_ID_interval  Next_same_ID_interval  
0   2.026558e-06                    NaN               0.018992  
1   9.536743e-07                    NaN               0.006927  
2   1.072884e-06                    NaN               0.011905  
3   1.012921e-03                    NaN               0.019993  
4   1.072884e-06                    NaN               0.098999  


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier



# Train Random Forest
rf = RandomForestClassifier(
    n_estimators=10,
    random_state=42
)

rf.fit(X_train, y_train)

# Predict
predictions = rf.predict(X_test)

In [9]:
predictions
y_test

0          R
1          R
2          R
3          R
4          R
          ..
5133796    R
5133797    R
5133798    R
5133799    R
5133800    R
Name: Attack, Length: 5133801, dtype: category
Categories (2, str): ['R', 'T']

In [10]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, predictions)
print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.9597


In [11]:
from sklearn.tree import export_text

print("Random Forest summary:")
print("n_estimators:", rf.n_estimators)
print("max_depth:", rf.max_depth)
print("min_samples_leaf:", rf.min_samples_leaf)
print("feature_importances:", dict(zip([
    "ID","Prev_ID","Next_ID","Former_ID","Latter_ID",
    "DLC","Prev_interval","Next_interval",
    "Prev_same_ID_interval","Next_same_ID_interval"
], rf.feature_importances_)))
print()

print("First tree structure (truncated):")
print(export_text(rf.estimators_[9], feature_names=[
    "ID","Prev_ID","Next_ID","Former_ID","Latter_ID",
    "DLC","Prev_interval","Next_interval",
    "Prev_same_ID_interval","Next_same_ID_interval"
], max_depth=10))

Random Forest summary:
n_estimators: 10
max_depth: None
min_samples_leaf: 1
feature_importances: {'ID': np.float64(0.3154897385538924), 'Prev_ID': np.float64(0.08030323339711692), 'Next_ID': np.float64(0.11401039806261473), 'Former_ID': np.float64(0.05272919770830128), 'Latter_ID': np.float64(0.06322965734858724), 'DLC': np.float64(0.0), 'Prev_interval': np.float64(0.034415800537543964), 'Next_interval': np.float64(0.06659505770215342), 'Prev_same_ID_interval': np.float64(0.20608205481184286), 'Next_same_ID_interval': np.float64(0.06714486187794726)}

First tree structure (truncated):
|--- Prev_interval <= 0.00
|   |--- ID <= 211.50
|   |   |--- ID <= 206.00
|   |   |   |--- class: 0.0
|   |   |--- ID >  206.00
|   |   |   |--- Prev_same_ID_interval <= 0.00
|   |   |   |   |--- Next_same_ID_interval <= 0.00
|   |   |   |   |   |--- class: 0.0
|   |   |   |   |--- Next_same_ID_interval >  0.00
|   |   |   |   |   |--- class: 1.0
|   |   |   |--- Prev_same_ID_interval >  0.00
|   |   |  

In [12]:
import joblib
hello()

joblib.dump(rf, "saved_models/random_forest_feature_vector_2.joblib")

NameError: name 'hello' is not defined

In [ ]:
rf_loaded = joblib.load("saved_models/random_forest_feature_vector_2.joblib")
print(rf_loaded)

RandomForestClassifier(n_estimators=10, random_state=42)


In [13]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

accuracy = accuracy_score(y_test, predictions)
precision = precision_score(y_test, predictions, average="weighted", zero_division=0)
recall = recall_score(y_test, predictions, average="weighted", zero_division=0)
f1 = f1_score(y_test, predictions, average="weighted", zero_division=0)

labels = list(rf.classes_)
cm = confusion_matrix(y_test, predictions, labels=labels)

cm_df = pd.DataFrame(cm, index=labels, columns=labels)

fpr = {}
fnr = {}
for i, label in enumerate(labels):
    tp = cm[i, i]
    fn = cm[i, :].sum() - tp
    fp = cm[:, i].sum() - tp
    tn = cm.sum() - tp - fn - fp
    fpr[label] = fp / (fp + tn) if (fp + tn) != 0 else 0.0
    fnr[label] = fn / (fn + tp) if (fn + tp) != 0 else 0.0

print(f"Accuracy: {accuracy:.4f}")
print(f"Weighted precision: {precision:.4f}")
print(f"Weighted recall: {recall:.4f}")
print(f"Weighted F1 score: {f1:.4f}")
print("\nClassification report:")
print(classification_report(y_test, predictions, labels=labels, zero_division=0))
print("Confusion matrix:")
print(cm_df)
print("\nFalse positive rate per class:")
for label, rate in fpr.items():
    print(f"  {label}: {rate:.4f}")
print("\nFalse negative rate per class:")
for label, rate in fnr.items():
    print(f"  {label}: {rate:.4f}")

Accuracy: 0.9597
Weighted precision: 0.9974
Weighted recall: 0.9597
Weighted F1 score: 0.9773

Classification report:
              precision    recall  f1-score   support

           R       1.00      0.96      0.98   5121499
           T       0.05      0.87      0.09     12302

    accuracy                           0.96   5133801
   macro avg       0.52      0.92      0.54   5133801
weighted avg       1.00      0.96      0.98   5133801

Confusion matrix:
         R       T
R  4916002  205497
T     1567   10735

False positive rate per class:
  R: 0.1274
  T: 0.0401

False negative rate per class:
  R: 0.0401
  T: 0.1274
